# EVE 310 - Lecture notebook: Linear regression traffic example

**Module 2 | Lecture follow-along**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/lectures/notebooks/linear-regression-traffic-example.ipynb)

This is the code shown in lecture for the traffic example. Run each cell as it comes up, so the output on your screen matches the one in lecture.

## Learning objectives

By the end of this notebook you will be able to:

1. Fit a straight line to data with NumPy (`np.polyfit`) and with scikit-learn (`LinearRegression`)
2. Predict with the fitted model and compute R²
3. Check a model with a predicted-vs-observed plot and a residual plot

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run one cell at a time, top to bottom, as it comes up in lecture: click the cell and press **Shift + Enter**.
3. Keep to the order. The scikit-learn section changes `x` and `y`, and the note there says what to do before you run an earlier cell again.
4. There is nothing to fill in and nothing to submit.

## Lecture 10 traffic example

Austin department of transportation needs to assess if it should add another lane to Mopac. To plan whether to add additional lane, we need to estimate the expected traffic volume in a typical day. The first step towards that is to estimate the number of trips taken by a typical household in Austin. You’ve conducted a survey polling the number of people in a household and the number of average trips taken each day.

| people | 1 | 2 | 3 | 4 | 5 | 1 | 2 | 3 |  4 | 5 | 1 | 2 |3 | 4 | 5 |
| ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| trips | 0.9 | 1.8 | 2.4 | 4.3 | 6.6 | 1.9 | 2.7 | 3.6 | 5.4 | 7.7 | 2.1 | 3.1 | 3.7 | 5.9 | 8.3 |

Find a model that estimates the number of trips as a function of size of the household.

Lab 05 used these same 15 households with a 70% / 30% train/test split. Here every fit uses all 15, so the intercept, slope, and R² are not the ones you saw in the lab.

In [ ]:
# Import libraries
import numpy as np #Data manipulation
import matplotlib.pyplot as plt # Visualization


In [ ]:
# create two arrays
people = np.array([1,2,3,4,5,1,2,3,4,5,1,2,3,4,5])
trips = np.array([0.9,1.8,2.4,4.3,6.6,1.9,2.7,3.6,5.4,7.7,2.1,3.1,3.7,5.9,8.3])
x = people
y = trips

Plot the data before fitting anything. Colab prints a line such as `Text(0, 0.5, 'number of trips')` above the plot. That is the value returned by the last command in the cell, not an error.

In [ ]:
# plot the data
f = plt.figure()
plt.plot(x, y, 'or')
plt.title('trips vs. people')
plt.xlabel('number of people')
plt.ylabel('number of trips')

## Regression with NumPy

`np.polyfit(x, y, 1)` fits a polynomial of degree 1, a straight line, by least squares. It returns the coefficients from the highest power down: the slope b1 first, then the intercept b0.

In [ ]:
# best fit polynomials y = b0 + b1*x
# We can obtain this line using Numpy. 
# The numpy function polyfit numpy.polyfit(x,y,deg) fits a polynomial of degree deg to points (x, y), 
# returning the polynomial coefficients that minimize the square error. 
# note that the coefficients are arranged as: bn, bn-1, ... b0

beta_coefficients = np.polyfit(x, y, 1)

In [ ]:
print(beta_coefficients)

In [ ]:
# plot
f = plt.figure(figsize = (10,7))
plt.plot(x, y, 'or')
plt.title('linear regression with numpy')
plt.xlabel('number of people')
plt.ylabel('number of trips')

# plot regression lines
plt.plot(x, beta_coefficients[0] * x + beta_coefficients[1], color='darkblue', linewidth=2)

# regression equations
plt.text(2, 6, 'y={:.2f}+{:.2f}*x'.format(beta_coefficients[1], beta_coefficients[0]), color='darkblue', size=12)

## Regression with scikit-learn

scikit-learn takes the input as a 2-D array with one row per observation, so this cell reshapes `x` and `y` from shape `(15,)` to `(15, 1)`. The comments mention a variable explorer, which is part of Spyder. In Colab, run `print(x.shape)` in a new cell before and after this one to see the dimensions change.

**If you go back:** after this cell `x` and `y` are 2-D, and `np.polyfit` in the NumPy section stops with `TypeError: expected 1D vector for x`. Run the cell that creates the two arrays again first.

In [ ]:
# Fitting a simple linear model using sklearn
from sklearn.linear_model import LinearRegression

# create linear regression object
# According to the Scikit-learn documentation, the input array should have a 2D array of the size (n_samples, n_features). 
# As such, we need to reshpae the input
# .reshape will come up a lot and you should remember it

x = x.reshape(-1, 1) # check the variable explorer to see how the dimensions change after running this command
y = y.reshape(-1,1) # check the variable explorer to see how the dimensions change after running this command

# creating an instance of a linear model to predict the number of trips 'lr_trips'
lm_trips = LinearRegression()

# fitting linear regression to the data (x,y); x = input, y = output
lm_trips.fit(x, y)

# get the slope and intercept of the line best fit
print(lm_trips.intercept_)

print(lm_trips.coef_)


`lm_trips.predict(x)` returns the model's estimate of the trips for every household in `x`.

In [ ]:
# get prediction
y_predict = lm_trips.predict(x) 

# plot 
# Plot outputs
f = plt.figure()
plt.plot(x, y, 'ob')  # plot observations
plt.plot(x, y_predict, color='black', linewidth=1) # plot prediction
plt.title('trips vs. people')
plt.xlabel('number of people')
plt.ylabel('number of trips')
plt.grid(True)

## Metrics to assess the model

R² compares the model's squared error with the total variation in `y`: R² = 1 − SSE/SST. The closer it is to 1, the better the fit. `r2_score` takes the observed values first, then the predicted values.

In [ ]:
# import library to evaluate predictions
from sklearn import metrics

print ('----------------')
print('R-squared')
R2 = metrics.r2_score(y,y_predict) 
print(R2)

## Model vs. observations plot

Each point is one household: observed trips on the horizontal axis, predicted trips on the vertical axis. A perfect model puts every point on the diagonal, where predicted = observed.

In [ ]:
# plot predicted vs. observedf = plt.figure()
plt.plot(y, y_predict, 'ob')  # plot observations
plt.title('predicted vs observed: linear')
plt.xlabel('observed values')
plt.ylabel('predicted values')
plt.grid(True)

## Residual plot

A residual is observed minus predicted. For a good model the residuals scatter around 0 with no pattern. Check whether these do.

In [ ]:
# plot residuals vs. predicted values
# linear model
residuals = y - y_predict

f = plt.figure()
plt.plot(y_predict,residuals, 'ob')  # plot observations
plt.title('residuals : linear')
plt.xlabel('predicted values')
plt.ylabel('residuals')
plt.grid(True)

## Commands used

- `np.polyfit(x, y, 1)`: least-squares line, returned as slope then intercept
- `LinearRegression()` and `.fit(x, y)`: create the scikit-learn model and fit it
- `.intercept_` and `.coef_`: the fitted intercept and slope
- `.predict(x)`: the model's predictions
- `metrics.r2_score(y, y_predict)`: R²